# watsonx Lab — Technical Track

**AI Insights from a Trusted Data Catalog**

Build a trusted catalog, a SQL (Presto) engine, a Milvus vector database, a
governed connection, and a text-to-SQL + charting pipeline on
**watsonx.data + watsonx.ai**.

This notebook spells out every step in code so you can see the internals. Run
the cells top to bottom. Where a step happens in the watsonx.data / watsonx.ai
**UI**, there's a markdown cell describing exactly what to click, followed by the
code that uses what you created.

> **Full written instructions:** see `labs/technical/README.md`.
> **Cost:** the Presto and Milvus engines bill while running. Start them early,
> and **stop both at the end** (last step).

## 1 · Install and authenticate

Run this first. It installs the SDK and signs you in. Fill in your region URL,
IBM Cloud API key, and watsonx.ai project ID (project → Manage → General).

In [ ]:
!pip install -q ibm-watsonx-ai pymilvus pypdf prestodb pandas matplotlib

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials

URL        = 'https://us-south.ml.cloud.ibm.com'   # your region
API_KEY    = '...'      # your IBM Cloud API key
PROJECT_ID = '...'      # watsonx.ai project ID

client = APIClient(Credentials(url=URL, api_key=API_KEY), project_id=PROJECT_ID)
print('authenticated:', client.version)

If the cell prints a version string, you're authenticated. If it raises an auth
error, re-check the API key and that `PROJECT_ID` matches the open project; the
region in `URL` must match where your project lives (e.g. `us-south`, `eu-de`,
`ca-tor`).

## 2 · watsonx.data — set up & explore (UI)

Do these in the watsonx.data console, then come back to the notebook. Create the
engines **first** so they reach *running* while you ingest the data.

1. **Trusted catalog** — Infrastructure manager → add/associate a catalog named
   **`iceberg_data`** (back it with the lab's storage bucket if prompted).
2. **SQL engine** — Add component → **Presto** (or Prestissimo). Name it
   `lab-presto`, smallest size, defaults, Create. Associate it with
   `iceberg_data`. Start it and move on while it warms up.
3. **Vector database** — Add component → **Milvus**. Name it `lab-milvus`,
   smallest size, Create. When running, open its connection details and copy the
   **host, port, user, password** — you'll paste them below.
4. **Ingest the table** — Data manager → in `iceberg_data` create a schema
   **`marine`** → **Create table from file** → upload `data/cod_landings_nl.csv`
   → name it **`cod_landings_nl`** (year/licensed_fishers = INT,
   landings_tonnes = DOUBLE, fishery_status = VARCHAR) → Finish.
5. **Federated SQL** — in the query workspace, select your Presto engine and run:

```sql
SELECT year, landings_tonnes, licensed_fishers, fishery_status
FROM iceberg_data.marine.cod_landings_nl
WHERE year BETWEEN 1955 AND 2024
ORDER BY year;
```

You should get one row per year, including the 1992 moratorium row. Presto is
querying the table **where it lives**, through the catalog — no copies, no ETL.

## 3 · watsonx.ai — connect to the AI (governed connection)

In your watsonx.ai project: **Assets → New asset → Connection → watsonx.data
(Presto)**. Supply the Presto engine's host, port, and credentials (from its
*Connect* panel), set SSL/HTTPS, name it `watsonx-data-lakehouse`, Test & Save.
This registers the catalogued source as a governed asset — read through access
controls, not by holding a copy.

Now point the Presto Python client at the same engine and define a small helper
that returns a dataframe.

In [ ]:
# Presto connection details — from the engine's "Connect" panel in watsonx.data
PRESTO_HOST = '...'
PRESTO_PORT = 443
PRESTO_USER = '...'
PRESTO_PWD  = '...'

In [ ]:
import prestodb, pandas as pd

presto = prestodb.dbapi.connect(
    host=PRESTO_HOST, port=PRESTO_PORT, user=PRESTO_USER,
    catalog='iceberg_data', schema='marine',
    http_scheme='https',
    auth=prestodb.auth.BasicAuthentication(PRESTO_USER, PRESTO_PWD))

def run_sql(sql):
    cur = presto.cursor(); cur.execute(sql)
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

run_sql('SELECT count(*) AS n FROM cod_landings_nl')

A single count row means the AI runtime is reading your catalogued table in
place. If auth fails, re-copy the credentials; if the host is unreachable,
confirm the engine is *running*.

## 4 · Load the PDF into your vector database (RAG)

There's no PDF-upload button in watsonx.data because turning a document into
vectors is a **watsonx.ai** job. From this notebook you extract the PDF text,
embed each passage with a watsonx.ai embedding model, and insert the vectors
into the Milvus you stood up.

First, read and chunk the PDF:

In [ ]:
from pypdf import PdfReader

reader = PdfReader('docs/cod_background.pdf')
pages  = [p.extract_text() or '' for p in reader.pages]
full   = '\n'.join(pages)

# simple fixed-size chunks (~500 chars) with a little overlap
chunks, step, size = [], 400, 500
for i in range(0, len(full), step):
    piece = full[i:i+size].strip()
    if len(piece) > 50: chunks.append(piece)
print(len(chunks), 'chunks from the PDF')

The 100-char overlap keeps sentences that straddle a boundary searchable from
either chunk. Now embed the chunks and insert them into your Milvus collection
(host/port/credentials from the Milvus connection details you copied earlier):

In [ ]:
MILVUS_HOST = '...'
MILVUS_PORT = '...'
MILVUS_USER = '...'
MILVUS_PWD  = '...'

In [ ]:
from ibm_watsonx_ai.foundation_models import Embeddings
from pymilvus import connections, Collection, FieldSchema, CollectionSchema, DataType

emb = Embeddings(model_id='ibm/slate-125m-english-rtrvr',
                 api_client=client, project_id=PROJECT_ID)
vectors = emb.embed_documents(texts=chunks)

connections.connect(host=MILVUS_HOST, port=MILVUS_PORT,
                    user=MILVUS_USER, password=MILVUS_PWD, secure=True)
schema = CollectionSchema([
    FieldSchema('id', DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema('text', DataType.VARCHAR, max_length=4000),
    FieldSchema('vector', DataType.FLOAT_VECTOR, dim=len(vectors[0]))])
col = Collection('cod_docs', schema)
col.insert([chunks, vectors])
col.create_index('vector', {'metric_type':'L2','index_type':'AUTOINDEX'})
col.load()
print('inserted', len(chunks), 'vectors into cod_docs')

Now search the PDF **by meaning**:

In [ ]:
q = emb.embed_query(text='what happened to the fishery in the early 1990s?')
hits = col.search([q], 'vector', {'metric_type':'L2'}, limit=3,
                  output_fields=['text'])
for hit in hits[0]:
    print(round(hit.distance, 2), hit.entity.get('text')[:140])

It returns the most relevant passages by **meaning**, not keyword match — a query
about the early 1990s surfaces the moratorium passage even if those exact words
never appear. The vectors landed in **your** governed Milvus — the document never
left your environment. If search returns nothing, confirm `col.load()` ran and
that the schema `dim` matches `len(vectors[0])`.

> **No-code alternative:** watsonx.ai's **Vector Index** asset does this
> point-and-click (upload PDF, pick the embedding model, point it at this same
> Milvus). That's the path the non-technical track uses — same thing with the lid
> on.

## 5 · Ask a question, get a visual

Granite turns a plain-English question into SQL and runs it on your catalogued
data; the model only ever sees the **schema and the question** — never the rows.

In [ ]:
from ibm_watsonx_ai.foundation_models import ModelInference

granite = ModelInference(model_id='ibm/granite-3-8b-instruct',
    api_client=client, project_id=PROJECT_ID,
    params={'decoding_method':'greedy', 'max_new_tokens':200})

schema   = ('cod_landings_nl(year INT, landings_tonnes DOUBLE, '
            'licensed_fishers INT, fishery_status VARCHAR)')
question = 'cod landings in tonnes for every year, ordered by year'
prompt = f'''Translate the question into ONE Presto SQL query.
Table: iceberg_data.marine.{schema}
Rules: SELECT only, no DDL/DML, add LIMIT 1000. Return ONLY the SQL.
Question: {question}'''

sql = granite.generate_text(prompt=prompt).strip().strip('`').strip()
print(sql)

**Validate before you run it — never execute model-generated SQL blind:**

In [ ]:
low = sql.lower()
assert low.startswith('select'), 'Not a SELECT'
assert not any(k in low for k in [' insert',' update',' delete',
       ' drop',' alter',' create',' merge']), 'Mutation blocked'

df = run_sql(sql)
df.head()

Those two asserts are the whole safety story: they guarantee the statement is
read-only before it touches the engine. In production you'd extend the check
(parse the SQL, allow-list tables, cap rows) — same principle: validate model
output, then run it.

### Let Granite generate the chart

You have a dataframe `df`. Ask Granite to write matplotlib for it, review the
code, then run it.

In [ ]:
prompt = f'''You are a Python data-viz assistant. A pandas dataframe `df`
has columns: {list(df.columns)}.
Write matplotlib code to plot landings_tonnes vs year as a line chart,
add a vertical marker and label at 1992 (the cod moratorium),
give it a clear title, and call plt.show(). Return ONLY code.'''
generated = granite.generate_text(prompt=prompt)
print(generated)

In [ ]:
import matplotlib.pyplot as plt
exec(generated)   # read it first, then run

The chart renders: the late-1960s peak (~800k tonnes), the decline, and the 1992
collapse marked on the axis. If `exec` errors, read the generated code — the
model occasionally references a column name that differs from `df.columns`; fix
it (or re-prompt with the exact column list) and run again.

> **Analytical follow-up:** change `question` in step 5 to something like
> *'average landings per decade and the peak year in each decade'*, re-run
> text-to-SQL, then let Granite re-plot the new dataframe. The whole pipeline
> flexes to a new question with no hand-written SQL or plotting code.

## 6 · Wrap-up & stop your engines

**What you built, and what it gives you beyond a DIY (Postgres + Jupyter + hosted
API) stack:**

| What you built | Beyond a DIY stack |
|---|---|
| A trusted catalog, queried in place | No ETL, no copies to secure; federate across sources with one engine |
| A vector DB loaded from a PDF | Structured SQL **and** document search under one set of controls |
| Text-to-SQL over a governed connection | The model sees only schema + question; rows stay in the catalog |
| Granite ran in the platform | IP-indemnified models beside the data — no export to an outside API |

**The sensitive-data pivot:** the cod data is public. Swap the table for genomics
data, student records, or unpublished research, and the PDF for a confidential
report — every step is identical, and nothing ever left your governed
environment or reached an outside model service.

> ### ⚠️ Stop your engines
> You created your own Presto and Milvus engines and they **bill while running**.
> Go back to watsonx.data → Infrastructure manager and **stop both now** to end
> billing. This is the last step.